# Praktikum SKLearn SVM untuk Klasifikasi
**Nama:** Soulthon Aji Mohammad Ramadhan  
**Platform:** Visual Studio Code (VS Code)  
**Algoritma:** Support Vector Machine (SVM)  
**Dataset:** `diabetes.csv` (Pima Indians Diabetes)

**Persiapan folder:** Simpan notebook ini dan file `diabetes.csv` di folder yang sama.  
**Tujuan:** Identifikasi dataset, preprocessing, pembagian data, pelatihan SVM, prediksi, dan evaluasi.

## 0. Instalasi Library
Jalankan perintah berikut di Terminal VS Code (cukup sekali, bukan di sel Python):

```bash
python -m pip install numpy pandas matplotlib seaborn scikit-learn jupyter ipykernel
```

Buka notebook ini di VS Code, pilih kernel Python, lalu jalankan sel dari atas ke bawah.

## 1. Import Library

Library digunakan untuk membaca dan mengolah dataset, membuat grafik, serta membangun dan mengevaluasi model klasifikasi SVM.

In [ ]:
# Mengimpor library untuk pengolahan data, visualisasi, dan algoritma SVM.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
print("Library berhasil diimpor.")

## 2. Membaca Dataset dari Folder Lokal
Pastikan `diabetes.csv` berada satu folder dengan notebook ini. Jika nama atau lokasinya berbeda, ubah nilai `file_path`.

Bagian ini menentukan lokasi file CSV, memeriksa keberadaan file, lalu membaca data ke dalam DataFrame agar dapat diproses dengan pandas.

In [ ]:
# Membaca file diabetes.csv setelah memastikan file tersedia di folder kerja.
file_path = "diabetes.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File '{file_path}' tidak ditemukan. "
        "Simpan diabetes.csv di folder yang sama dengan notebook."
    )

df = pd.read_csv(file_path)
print("Dataset berhasil dibaca.")
display(df.head())

## 3. Identifikasi Dataset

Tahap identifikasi bertujuan mengetahui jumlah baris dan kolom, tipe data, ringkasan statistik, nilai kosong, serta data duplikat.

In [ ]:
# Menampilkan ukuran, struktur, statistik, nilai kosong, dan data duplikat.
print("Ukuran dataset (baris, kolom):", df.shape)
print("\nInformasi dataset:")
df.info()

print("\nStatistik deskriptif:")
display(df.describe())

print("\nJumlah nilai kosong:")
display(df.isnull().sum())

print("\nJumlah data duplikat:", df.duplicated().sum())

## 4. Preprocessing Data
Untuk dataset Pima Indians Diabetes, nilai 0 pada Glucose, BloodPressure, SkinThickness, Insulin, dan BMI umumnya diperlakukan sebagai nilai tidak valid/missing pada latihan ini. Ganti dengan median. Pastikan asumsi ini sesuai dengan dataset yang digunakan.

Preprocessing membersihkan data sebelum pemodelan. Nilai 0 pada kolom tertentu diubah menjadi nilai kosong, lalu diisi median. Data duplikat juga dihapus.

In [ ]:
# Mengganti nilai 0 yang dianggap tidak valid, mengisi nilai kosong dengan median,
# lalu menghapus baris duplikat.
kolom_nol = [
    kolom for kolom in
    ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    if kolom in df.columns
]

for kolom in kolom_nol:
    df[kolom] = df[kolom].replace(0, np.nan)
    df[kolom] = df[kolom].fillna(df[kolom].median())

df = df.drop_duplicates()

print("Nilai kosong setelah preprocessing:")
display(df.isnull().sum())
print("Ukuran data setelah preprocessing:", df.shape)

## 5. Menentukan Fitur (X) dan Target (y)

Fitur (X) adalah kolom-kolom yang digunakan sebagai bahan prediksi, sedangkan target (y) adalah kolom Outcome yang berisi label kelas diabetes.

In [ ]:
# Memisahkan kolom prediktor (X) dari kolom target/label (y).
if 'Outcome' not in df.columns:
    raise ValueError("Kolom 'Outcome' tidak ditemukan. Periksa nama kolom dataset.")

X = df.drop(columns=['Outcome'])
y = df['Outcome']

print("Kolom fitur:", list(X.columns))
print("\nDistribusi label:")
display(y.value_counts().sort_index())

## 6. Membagi Data Training dan Testing
Data dibagi 80% untuk training dan 20% untuk testing. `stratify=y` mempertahankan proporsi kelas.

Data dibagi menjadi data training untuk melatih model dan data testing untuk menguji model. Pembagian 80:20 digunakan pada praktikum ini.

In [ ]:
# Membagi dataset menjadi 80% data training dan 20% data testing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])

## 7. Normalisasi Data
Scaler dipelajari dari data training saja untuk menghindari data leakage.

Normalisasi menyamakan skala nilai fitur agar perbedaan rentang angka tidak terlalu memengaruhi proses pembelajaran SVM.

In [ ]:
# Menghitung skala dari data training, lalu menerapkannya ke training dan testing.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Normalisasi selesai.")

## 8. Membuat dan Melatih Model SVM

Model Support Vector Machine dibuat dengan kernel linear, kemudian dilatih menggunakan data training yang sudah dinormalisasi.

In [ ]:
# Membuat SVM dengan kernel linear dan melatihnya menggunakan data training.
model = SVC(kernel='linear', C=1.0)
model.fit(X_train_scaled, y_train)

print("Model SVM berhasil dilatih.")

## 9. Melakukan Prediksi

Model yang telah dilatih digunakan untuk memperkirakan kelas pada data testing. Hasil prediksi dibandingkan dengan label aktual.

In [ ]:
# Memprediksi kelas data testing dan menampilkan contoh hasil aktual vs prediksi.
y_pred = model.predict(X_test_scaled)

hasil = pd.DataFrame({
    'Aktual': y_test.to_numpy(),
    'Prediksi': y_pred
})
display(hasil.head(10))

## 10. Menghitung Akurasi

Akurasi menunjukkan persentase jumlah prediksi yang benar dibandingkan seluruh data testing.

In [ ]:
# Menghitung persentase prediksi yang sesuai dengan label sebenarnya.
akurasi = accuracy_score(y_test, y_pred)
print(f"Akurasi model SVM: {akurasi * 100:.2f}%")

## 11. Confusion Matrix

Confusion matrix memperlihatkan jumlah prediksi benar dan salah untuk setiap kelas, sehingga kesalahan klasifikasi dapat diamati.

In [ ]:
# Membuat confusion matrix untuk melihat hasil klasifikasi tiap kelas.
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negatif (0)', 'Positif (1)'],
    yticklabels=['Negatif (0)', 'Positif (1)']
)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix - SVM')
plt.tight_layout()
plt.show()

## 12. Classification Report

Classification report menampilkan precision, recall, F1-score, dan jumlah data pada setiap kelas untuk menilai kinerja model.

In [ ]:
# Menampilkan precision, recall, F1-score, dan support untuk masing-masing kelas.
print(classification_report(
    y_test,
    y_pred,
    target_names=['Negatif Diabetes (0)', 'Positif Diabetes (1)'],
    zero_division=0
))

## 13. Kesimpulan
Model SVM digunakan untuk mengklasifikasikan data menjadi dua kelas berdasarkan fitur dataset. Evaluasi dilakukan menggunakan akurasi, confusion matrix, precision, recall, dan F1-score.

**Lengkapi setelah menjalankan notebook:** tuliskan nilai akurasi yang diperoleh dan jelaskan hasil precision, recall, serta F1-score.

*Catatan: hasil model untuk latihan ini bukan alat diagnosis medis.*